In [1]:
!pip install torch transformers datasets peft trl accelerate bitsandbytes swanlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.2/371.2 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 20.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
import json
import torch
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import gc
import os
import shutil
import json
import time
from datetime import datetime
import math
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from torch.profiler import profile, record_function, ProfilerActivity, schedule
from collections import defaultdict
import pynvml
import threading


from transformers.utils import is_flash_attn_2_available
from transformers import get_cosine_schedule_with_warmup

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
TRAIN_DATA_PATH = Path('/content/drive/MyDrive/HPML Project/SFT/data/gsm8k_sft_train.json')
VAL_DATA_PATH   = Path('/content/drive/MyDrive/HPML Project/SFT/data/gsm8k_sft_val.json')


MODEL_ID   = 'Qwen/Qwen2.5-3B-Instruct'
CUTOFF_LEN = 512
dataset_name = "openai/gsm8k"
output_dir = "./qwen2.5-3b-distill-student"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
train_records = json.loads(TRAIN_DATA_PATH.read_text(encoding='utf-8'))

In [7]:
def tokenize(record):
    messages = [
        {"role": "user",      "content": record["instruction"]},
        {"role": "assistant", "content": record["output"]},
    ]

    full = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        truncation=True,
        max_length=CUTOFF_LEN,
    )

    prompt_only = tokenizer.apply_chat_template(
        [{"role": "user", "content": record["instruction"]}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    )

    input_ids = full["input_ids"]
    prompt_len = len(prompt_only["input_ids"])

    labels = [-100] * prompt_len + input_ids[prompt_len:]

    return {
        "input_ids": input_ids,
        "labels":    labels,
    }

tokenized = [tokenize(r) for r in train_records]

In [8]:
max_len = max(len(x["input_ids"]) for x in tokenized)

def pad(seq, pad_id, length):
    return seq + [pad_id] * (length - len(seq))

padded = [
    {
        "input_ids": pad(x["input_ids"], tokenizer.pad_token_id, max_len),
        "labels":    pad(x["labels"],    -100,                   max_len),
    }
    for x in tokenized
]

In [9]:
distill_dataset = Dataset.from_list(padded)
distill_dataset.set_format(type="torch", columns=["input_ids", "labels"])

print(f"distill_dataset size : {len(distill_dataset)}")
print(f"input_ids shape      : {distill_dataset[0]['input_ids'].shape}")
print(f"labels shape         : {distill_dataset[0]['labels'].shape}")

distill_dataset size : 2700
input_ids shape      : torch.Size([386])
labels shape         : torch.Size([386])


# Separate Profiling

## Model Set Up

In [10]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    # task_type="CAUSAL_LM",
)

In [11]:
try: del model_distill
except: pass
gc.collect()
torch.cuda.empty_cache()

model_distill = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2" if is_flash_attn_2_available() else "eager",
)
model_distill = get_peft_model(model_distill, lora_config)
model_distill.print_trainable_parameters()
# save_path = "./qwen2.5-3B-Distill-Orig"
# last_model_path = save_path

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031


## GPU Utilization Polling Thread

In [12]:
class GPUPoller:
    def __init__(self, handle, interval=0.05):  # 50ms polling
        self.handle = handle
        self.interval = interval
        self.samples = []
        self._stop = threading.Event()
        self._t = None

    def start(self):
        self.samples = []  # reset each time
        self._stop.clear()
        self._t = threading.Thread(target=self._poll, daemon=True)
        self._t.start()

    def stop(self):
        self._stop.set()
        self._t.join()

    def _poll(self):
        while not self._stop.is_set():
            u = pynvml.nvmlDeviceGetUtilizationRates(self.handle)
            self.samples.append(u.gpu)
            time.sleep(self.interval)

    @property
    def mean(self):
        return sum(self.samples) / len(self.samples) if self.samples else 0

    @property
    def peak(self):
        return max(self.samples) if self.samples else 0

## SFT & Profiling Set Up

In [13]:
BATCH_SIZE = 16
NUM_WORKERS = 4
LR = 2e-05
NUM_EPOCHS = 1
PIN_MEMORY = False
PROFILE_STEPS = 24
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
GRADIENT_CHECKPOINT = True

prof_schedule = schedule(wait=1, warmup=1, active=10, repeat=2)

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
poller = GPUPoller(handle, interval=0.05)

In [14]:
distill_dataset.set_format(type="torch", columns=["input_ids", "labels"])
train_loader = DataLoader(distill_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

optimizer = torch.optim.AdamW(
    model_distill.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    eps=1e-8,
    fused=True,
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=int(WARMUP_RATIO * PROFILE_STEPS),
    num_training_steps=PROFILE_STEPS
)


STAGES = ["data_loading", "host_to_device", "forward", "backward", "optimizer"]

if GRADIENT_CHECKPOINT:
    model_distill.gradient_checkpointing_enable()


In [15]:
loader_iter = iter(train_loader)
batch = next(loader_iter)
print("Batch keys:", batch.keys())
print("Batch shapes:", {k: v.shape for k, v in batch.items()})

Batch keys: dict_keys(['input_ids', 'labels'])
Batch shapes: {'input_ids': torch.Size([16, 386]), 'labels': torch.Size([16, 386])}


In [16]:
step_records = []


with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
    with_flops=True,
) as prof:
    model_distill.train()
    loader_iter = iter(train_loader)

    for i in range(PROFILE_STEPS):
        stage_times = {}
        torch.cuda.reset_peak_memory_stats()
        poller.start()

        # -- data loading --
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with record_function("data_loading"):
            batch = next(loader_iter)
        torch.cuda.synchronize()
        stage_times["data_loading"] = time.perf_counter() - t0

        # -- host to device --
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with record_function("host_to_device"):
            batch = {k: v.to(device=model_distill.device) for k, v in batch.items()}
        torch.cuda.synchronize()
        stage_times["host_to_device"] = time.perf_counter() - t0

        # -- forward --
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with record_function("forward"):
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model_distill(**batch)
                loss = outputs.loss
        torch.cuda.synchronize()
        stage_times["forward"] = time.perf_counter() - t0

        # -- backward --
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with record_function("backward"):
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss.backward()
        torch.cuda.synchronize()
        stage_times["backward"] = time.perf_counter() - t0

        # -- optimizer --
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with record_function("optimizer"):
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        torch.cuda.synchronize()
        stage_times["optimizer"] = time.perf_counter() - t0

        poller.stop()

        peak_vram = torch.cuda.max_memory_allocated() / 1024**3
        reserved_vram = torch.cuda.max_memory_reserved() / 1024**3

        step_records.append({
            "step": i,
            "lr": scheduler.get_last_lr()[0],
            "stage_times": stage_times,
            "mean_gpu_util_pct": poller.mean,
            "peak_gpu_util_pct": poller.peak,
            "peak_vram_gb": peak_vram,
            "reserved_vram_gb": reserved_vram,
        })

        prof.step()

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


In [17]:
# PyTorch profiler kernel-level table
print("\n" + "="*80)
print("PYTORCH PROFILER — CUDA KERNEL BREAKDOWN (top 20 by cuda_time_total)")
print("="*80)
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=20
))

# per-stage averages across steps
print("\n" + "="*80)
print("PER-STAGE TIME BREAKDOWN (avg over steps, in ms)")
print("="*80)

stage_totals = defaultdict(float)
for rec in step_records:
    for stage, t in rec["stage_times"].items():
        stage_totals[stage] += t

n = len(step_records)
total_avg = sum(stage_totals.values()) / n

print(f"{'Stage':<20} {'Avg (ms)':>10} {'% of step':>12}")
print("-" * 44)
for stage in STAGES:
    avg_ms = stage_totals[stage] / n * 1000
    pct = (stage_totals[stage] / n) / total_avg * 100
    print(f"{stage:<20} {avg_ms:>10.2f} {pct:>11.1f}%")
print("-" * 44)
print(f"{'TOTAL STEP':<20} {total_avg*1000:>10.2f} {'100.0%':>12}")

# GPU utilization proxy (gpu_time / wall_time)
print("\n" + "="*80)
print("GPU UTILIZATION (NVML polling)")
print("="*80)
print(f"{'Step':<8} {'Mean Util %':>12} {'Peak Util %':>12} {'Samples':>8} {'Wall time (ms)':>15}")
print("-" * 60)

for rec in step_records:
    wall_ms = sum(rec["stage_times"].values()) * 1000

    print(
        f"{rec['step']:<8}"
        f"{rec['mean_gpu_util_pct']:>11.1f}%"
        f"{rec['peak_gpu_util_pct']:>11.1f}%"
        # f"{rec['n_util_samples']:>8}"
        f"{wall_ms:>15.2f}"
    )

# VRAM
print("\n" + "="*80)
print("PEAK VRAM (per step)")
print("="*80)
print(f"{'Step':<8} {'Peak alloc (GB)':>16} {'Peak reserved (GB)':>20}")
print("-" * 46)
for rec in step_records:
    print(f"{rec['step']:<8} {rec['peak_vram_gb']:>16.3f} {rec['reserved_vram_gb']:>20.3f}")

# memory traffic from profiler
print("\n" + "="*80)
print("MEMORY TRAFFIC — TOP OPS BY SELF CUDA MEMORY (profiler)")
print("="*80)
print(prof.key_averages().table(
    sort_by="self_cuda_memory_usage",
    row_limit=15
))

# export for TensorBoard
current_run = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = f"/content/drive/MyDrive/HPML Project/SFT/profiler/{current_run}"
os.makedirs(run_dir, exist_ok=True)

prof.export_chrome_trace(f"{run_dir}/trace.json")
print(f"\nTrace exported to {run_dir}/trace.json")


PYTORCH PROFILER — CUDA KERNEL BREAKDOWN (top 20 by cuda_time_total)
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  Total GFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
       autograd::engine::evaluate_function: MmBackward0         0.42%     235.051ms        22.27%       12.394s     795.721u

In [18]:
config = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "lr": LR,
    "num_epochs": NUM_EPOCHS,
    "pin_memory": PIN_MEMORY,
    "profile_steps": PROFILE_STEPS,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "gradient_checkpoint": GRADIENT_CHECKPOINT,
    "prof_schedule": {"wait": 1, "warmup": 1, "active": 10, "repeat": 2},
}

output = {
    "config": config,
    "steps": step_records,
}


with open(f"{run_dir}/step_records.json", "w") as f:
    json.dump(output, f, indent=2)